# 📄 RAG Chatbot — Chat with Any PDF

**What this notebook does:**
- Loads a PDF and splits it into smart chunks
- Embeds each chunk using OpenAI and stores in a FAISS vector store
- Builds a RAG chain that retrieves relevant chunks and answers questions
- Supports **multi-turn conversation** (chat history aware)
- Shows **source page numbers** for every answer

---
**Requirements:** OpenAI API key + a PDF file path

**Flow:**
```
PDF → Split → Embed → FAISS Store
                          ↓
Question → Retriever → LLM (GPT-4o-mini) → Answer + Sources
```

In [ ]:
# ============================================================
# CELL 1 — Install all required packages
# Run this once, then restart kernel if needed
# ============================================================

!pip install -U \
    langchain \
    langchain-community \
    langchain-openai \
    langchain-text-splitters \
    pypdf \
    faiss-cpu \
    python-dotenv

In [ ]:
# ============================================================
# CELL 2 — Imports & Configuration
# ============================================================

import os
from dotenv import load_dotenv

# --- Load from .env file (recommended) ---
# Create a .env file with:
#   OPENAI_API_KEY=sk-...
#   PDF_PATH=your_file.pdf
load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_API_KEY_HERE")
PDF_PATH       = os.environ.get("PDF_PATH", "your_file.pdf")

# --- OR set directly here (not recommended for sharing) ---
# OPENAI_API_KEY = "sk-..."
# PDF_PATH = "/path/to/your/file.pdf"

print(f"✅ API Key loaded: {'Yes' if OPENAI_API_KEY != 'YOUR_API_KEY_HERE' else '❌ Not set!'}")
print(f"📄 PDF Path: {PDF_PATH}")
print(f"📁 File exists: {'✅ Yes' if os.path.exists(PDF_PATH) else '❌ No — check your path!'}")

In [ ]:
# ============================================================
# CELL 3 — Load PDF & Split into Chunks
# ============================================================
#
# WHY SPLIT?  LLMs have a context window limit.
# We split the PDF into small overlapping chunks so we can
# retrieve only the RELEVANT pieces for each question.
#
# chunk_size=1000   → each chunk ~1000 characters
# chunk_overlap=150 → chunks overlap by 150 chars
#                     (avoids cutting mid-sentence context)
# ============================================================

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Step 1: Load the PDF (each page becomes a Document)
print("📥 Loading PDF...")
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"   → Loaded {len(pages)} pages")

# Step 2: Split each page into smaller chunks
print("✂️  Splitting into chunks...")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)
splits = splitter.split_documents(pages)
print(f"   → Created {len(splits)} chunks")

# Step 3: Enrich metadata so we can trace back answers later
print("🏷️  Attaching metadata to chunks...")
for i, chunk in enumerate(splits):
    chunk.metadata["chunk_id"]    = i                              # unique chunk number
    chunk.metadata["source_file"] = os.path.basename(PDF_PATH)    # filename only
    # chunk.metadata["page"] already set by PyPDFLoader (0-indexed)

print(f"\n✅ Done! Sample chunk preview:")
print("-" * 60)
print(f"Chunk ID  : {splits[0].metadata['chunk_id']}")
print(f"Page      : {splits[0].metadata.get('page', 'N/A')}")
print(f"Source    : {splits[0].metadata['source_file']}")
print(f"Content   : {splits[0].page_content[:300]}...")

In [ ]:
# ============================================================
# CELL 4 — Embed Chunks & Build FAISS Vector Store
# ============================================================
#
# WHY EMBEDDINGS?
# We convert each chunk into a vector (list of numbers).
# Similar meaning → similar vectors → easy to find relevant chunks.
#
# WHY FAISS?
# FAISS (Facebook AI Similarity Search) stores these vectors
# locally and does fast nearest-neighbor search.
# No external database needed!
#
# Model: text-embedding-3-small → cheap, accurate, 1536 dimensions
# ============================================================

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

print("🔢 Creating embeddings and building vector store...")
print("   (This may take 10–60s depending on PDF size)")

embeddings = OpenAIEmbeddings(
    api_key=OPENAI_API_KEY,
    model="text-embedding-3-small"   # cost-effective and powerful
)

# This embeds ALL chunks and stores them in memory
vectorstore = FAISS.from_documents(splits, embeddings)

# Retriever: fetch top-4 most relevant chunks per question
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"✅ Vector store ready with {vectorstore.index.ntotal} vectors stored!")

# Optional: save the index to disk so you don't re-embed every time
# vectorstore.save_local("faiss_index")
# To reload: vectorstore = FAISS.load_local("faiss_index", embeddings)

In [ ]:
# ============================================================
# CELL 5 — Build the RAG Chain (History-Aware)
# ============================================================
#
# Two-stage chain:
#
# STAGE 1 — History-Aware Retriever
#   Takes the chat history + new question
#   → Rewrites question as a standalone (no pronouns like "it", "that")
#   → Retrieves relevant chunks from vector store
#
# STAGE 2 — Question-Answer Chain
#   Takes retrieved chunks + question
#   → LLM reads the chunks and writes a grounded answer
#   → Refuses to hallucinate if the answer isn't in the PDF
# ============================================================

from langchain_openai import ChatOpenAI
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# The LLM used for both rewriting questions and answering
llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model="gpt-4o-mini",
    temperature=0            # 0 = deterministic/factual answers
)

# ── Stage 1 Prompt: Rewrite question as standalone ──────────
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """Given the chat history and the latest user question \
(which may reference earlier messages), rewrite it as a \
fully self-contained, standalone question. \
Do NOT answer it. Only rewrite if needed; otherwise return it as-is."""),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

# ── Stage 2 Prompt: Answer grounded in PDF context ──────────
qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a precise and helpful assistant for answering questions \
about a PDF document. \

RULES:
- Use ONLY the information in the retrieved context below.
- If the answer is not in the context, say: "I don't find this in the document."
- Keep answers concise and structured.
- Mention page numbers when available.

Context:
{context}"""),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# ── Combine both stages into one RAG chain ──────────────────
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

print("✅ RAG chain is ready!")

In [ ]:
# ============================================================
# CELL 6 — Add Memory / Chat History
# ============================================================
#
# RunnableWithMessageHistory wraps the RAG chain and
# automatically:
#   1. Loads previous messages from session store
#   2. Passes them as chat_history to the chain
#   3. Saves the new Q&A back to the session
#
# session_id = a unique string per conversation
# Change it to start a fresh conversation.
# ============================================================

from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# In-memory store (cleared when kernel restarts)
# For persistence, replace with SQLite or Redis-based history
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """Return (or create) chat history for a given session."""
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",         # key for new question
    history_messages_key="chat_history", # key for past messages
    output_messages_key="answer",        # key for the answer
)

print("✅ Memory layer attached. Ready to chat!")
print("💡 Tip: Change SESSION_ID below to start a fresh conversation.")

In [ ]:
# ============================================================
# CELL 7 — Ask Your First Question
# ============================================================

SESSION_ID = "my_session_1"   # change to reset memory

def ask(question: str, session_id: str = SESSION_ID) -> None:
    """
    Ask a question and print the answer with source citations.
    """
    print(f"\n{'='*60}")
    print(f"❓ Question: {question}")
    print(f"{'='*60}")

    result = conversational_rag_chain.invoke(
        {"input": question},
        config={"configurable": {"session_id": session_id}}
    )

    # ── Print the answer ────────────────────────────────────
    print(f"\n💬 Answer:\n")
    print(result["answer"])

    # ── Print the source chunks used ────────────────────────
    print(f"\n📚 Sources retrieved:")
    print("-" * 60)
    for doc in result.get("context", []):
        page   = doc.metadata.get("page", "?")           # 0-indexed from PyPDFLoader
        chunk  = doc.metadata.get("chunk_id", "?")
        source = doc.metadata.get("source_file", "?")
        print(f"📄 {source}  |  Page {int(page)+1}  |  Chunk #{chunk}")
        print(f"   {doc.page_content[:200].strip()}...")
        print()


# ── Ask your first question ─────────────────────────────────
ask("What is this document about? Summarize it briefly.")

In [ ]:
# ============================================================
# CELL 8 — Follow-up Questions (Chat History is Remembered)
# ============================================================
# The chain automatically uses previous Q&A to understand
# pronouns like "it", "that topic", "the second one", etc.
# ============================================================

ask("Can you explain the main concept in simpler terms?")

In [ ]:
# ============================================================
# CELL 9 — Your Custom Questions
# ============================================================
# Replace the question strings below with your own.
# You can call ask() as many times as you like.
# ============================================================

ask("What are the key conclusions or findings?")
# ask("Who are the authors?")
# ask("What methodology was used?")
# ask("List the main sections of the document.")

In [ ]:
# ============================================================
# CELL 10 — Utilities & Debugging
# ============================================================

# ── View full chat history for this session ──────────────────
print("📜 Chat history for session:", SESSION_ID)
print("-" * 60)
history = get_session_history(SESSION_ID)
for msg in history.messages:
    role = "👤 User" if msg.type == "human" else "🤖 AI"
    print(f"{role}: {msg.content[:200]}")
    print()

# ── Reset session (start fresh) ──────────────────────────────
# store.clear()   # clears ALL sessions
# store.pop(SESSION_ID, None)  # clears only this session

# ── Save FAISS index to disk (avoid re-embedding) ────────────
# vectorstore.save_local("faiss_index")
# To reload later:
# vectorstore = FAISS.load_local("faiss_index", embeddings,
#                                allow_dangerous_deserialization=True)

# ── Test retrieval manually (what chunks get fetched?) ───────
# test_docs = retriever.invoke("your test question here")
# for d in test_docs:
#     print(d.metadata, d.page_content[:300])